# 03: 3D Gaussian Splatting 学習

先頭セルの `SCENE_NAME` を 02 と同じ名前に設定して実行してください。

学習時間: T4 GPU で約 45〜90 分（30000 iter）。

In [ ]:
# ========== ユーザー設定（ここだけ変更） ==========
SCENE_NAME = "my_scene"
ITERATIONS  = 30000     # 短縮したいなら 7000
# ================================================

from google.colab import drive
from pathlib import Path
import torch

drive.mount("/content/drive")

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

DRIVE_ROOT   = Path("/content/drive/MyDrive/gaussian_splatting")
DATA_DIR     = DRIVE_ROOT / "output" / SCENE_NAME
LOCAL_OUTPUT = "/content/gs_output"
OUTPUT_PLY   = DATA_DIR / "point_cloud" / f"iteration_{ITERATIONS}" / "point_cloud.ply"

assert (DATA_DIR / "sparse" / "0" / "cameras.bin").exists(), (
    "COLMAPの結果が見つかりません。02_colmap_sfm.ipynb を先に実行してください。"
)
assert (DATA_DIR / "images").exists(), "images ディレクトリが見つかりません。"

imgs = list((DATA_DIR / "images").iterdir())
print(f"画像数: {len(imgs)}")
print(f"データ: {DATA_DIR}")

In [ ]:
import subprocess, os

if not os.path.isdir("/content/gaussian-splatting"):
    print("Cloning gaussian-splatting...")
    subprocess.run([
        "git", "clone", "--depth", "1", "--recursive",
        "https://github.com/graphdeco-inria/gaussian-splatting",
        "/content/gaussian-splatting"
    ], check=True)

try:
    import diff_gaussian_rasterization
    print("CUDA extensions already installed")
except ImportError:
    print("Building CUDA extensions...")
    subprocess.run([
        "pip", "install",
        "/content/gaussian-splatting/submodules/diff-gaussian-rasterization",
        "--quiet"
    ], check=True)
    subprocess.run([
        "pip", "install",
        "/content/gaussian-splatting/submodules/simple-knn",
        "--quiet"
    ], check=True)

subprocess.run(["pip", "install", "plyfile", "tqdm", "--quiet"], check=True)
print("準備完了")

In [ ]:
import subprocess

cmd = [
    "python", "/content/gaussian-splatting/train.py",
    "--source_path", str(DATA_DIR),
    "--model_path",  LOCAL_OUTPUT,
    "--iterations",  str(ITERATIONS),
    "--eval",
    # VRAM 不足時は以下のコメントを外す:
    # "--resolution", "2",
    # "--densify_until_iter", "10000",
]

print("学習コマンド:\n" + " ".join(cmd))
print(f"\n学習開始（{ITERATIONS} iter）...\n")

proc = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
for line in iter(proc.stdout.readline, ""):
    print(line, end="", flush=True)
proc.wait()

if proc.returncode != 0:
    raise RuntimeError(f"train.py が異常終了しました (code={proc.returncode})")
print("\n学習完了!")

In [ ]:
import shutil
from pathlib import Path

local_ply_dir = Path(LOCAL_OUTPUT) / "point_cloud"
dst_ply_dir   = DATA_DIR / "point_cloud"

if local_ply_dir.exists():
    shutil.copytree(str(local_ply_dir), str(dst_ply_dir), dirs_exist_ok=True)
    for f in Path(LOCAL_OUTPUT).iterdir():
        if f.is_file():
            shutil.copy2(str(f), str(DATA_DIR / f.name))

ply_path = dst_ply_dir / f"iteration_{ITERATIONS}" / "point_cloud.ply"
if ply_path.exists():
    size_mb = ply_path.stat().st_size / 1e6
    print(f"出力: {ply_path}")
    print(f"サイズ: {size_mb:.1f} MB")
    print("\n=== SuperSplat で閲覧 ===")
    print("1. https://playcanvas.com/supersplat/editor を開く")
    print("2. [Open] ボタンから point_cloud.ply を選択")
    print(f"   Drive パス: MyDrive/gaussian_splatting/output/{SCENE_NAME}/")
    print(f"              point_cloud/iteration_{ITERATIONS}/point_cloud.ply")
else:
    print(f"WARNING: PLY ファイルが見つかりません: {ply_path}")

In [ ]:
# PLY をローカルにダウンロードしたい場合に実行
from google.colab import files
import shutil
from pathlib import Path

ply_path = DATA_DIR / "point_cloud" / f"iteration_{ITERATIONS}" / "point_cloud.ply"
local_copy = Path(f"/content/point_cloud_{SCENE_NAME}.ply")
shutil.copy2(str(ply_path), str(local_copy))
files.download(str(local_copy))